# tuned - Kaggle smoke run
Prereqs: phone-verified account, Accelerator = **GPU T4 x2** (never P100), Internet **On**,
`HF_TOKEN` added under Add-ons -> Secrets. Set `MODE` below, then Run All
(SAVETEST interactively first; SMOKE via *Save & Run All* in the background).

In [ ]:
MODE = "SAVETEST"  # SAVETEST (4-step save/push gate) | SMOKE (60 steps) | RESUME
CONFIG = "configs/law_v1.yaml"  # escape hatch: configs/law_v1_qwen.yaml (see runbook)

import os, subprocess

os.environ["CUDA_VISIBLE_DEVICES"] = "0"        # single-T4 training (spec: DDP deferred)
os.environ["HF_HOME"] = "/tmp/hf_cache"          # scratch, NOT the 20GB persisted /kaggle/working
os.environ["UNSLOTH_STABLE_DOWNLOADS"] = "1"     # hub timeouts + skip the xet path
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "0"    # must be PRESENT and "0": unsloth_zoo force-enables
# hf_transfer whenever the var is absent (independently of UNSLOTH_STABLE_DOWNLOADS), and that
# fast path has no retry/resume and can stall silently at 90-95%
os.environ["HF_HUB_DISABLE_XET"] = "1"           # notebook process gets the same plain-http,
os.environ["HF_HUB_ETAG_TIMEOUT"] = "30"         # timeout-guarded download path the child gets
os.environ["HF_HUB_DOWNLOAD_TIMEOUT"] = "30"     # from unsloth_zoo
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1" # \r progress spam never renders in batch logs

gpus = subprocess.run(["nvidia-smi", "-L"], capture_output=True, text=True).stdout
print(gpus)
assert gpus.count("T4") == 2, "Expected 2x T4 - Settings -> Accelerator -> 'GPU T4 x2'"
print(subprocess.run(["df", "-h", "/tmp", "/kaggle/working"], capture_output=True, text=True).stdout)

In [ ]:
%cd /tmp
!rm -rf /tmp/tuned
!git clone --depth 1 https://github.com/Anant-T/Tuned /tmp/tuned
%cd /tmp/tuned

In [ ]:
import subprocess

subprocess.run(["pip", "install", "-q", "uv"], check=True)
r = subprocess.run(["uv", "pip", "install", "--system", "-e", ".[dev,train]"])
assert r.returncode == 0, "dependency install failed - do not continue on a broken env"

In [ ]:
import os
from pathlib import Path

tok = None
for f in Path("/kaggle/input").rglob("token.txt"):
    tok = f.read_text().strip()
    break
if tok is None:
    mounts = [str(p) for p in Path("/kaggle/input").rglob("*")][:20] if Path("/kaggle/input").exists() else "no /kaggle/input"
    print(f"token.txt not found anywhere under /kaggle/input; tree sample: {mounts}")
    try:
        from kaggle_secrets import UserSecretsClient

        tok = UserSecretsClient().get_secret("HF_TOKEN")
    except Exception as exc:
        raise SystemExit(
            "No HF token available: dataset 'tuned-token' not mounted AND "
            f"UI secret unavailable ({type(exc).__name__})."
        ) from exc
os.environ["HF_TOKEN"] = tok
print("HF token loaded (not printed).")

In [ ]:
from importlib.metadata import version

for pkg in ("torch", "transformers", "trl", "unsloth", "bitsandbytes", "peft", "hf_transfer"):
    try:
        print(f"{pkg}=={version(pkg)}")
    except Exception:
        print(f"{pkg}: NOT INSTALLED")

import subprocess

assert subprocess.run(["python", "-m", "pytest", "tests/", "-q"]).returncode == 0, "tests failed - fix before burning GPU quota"

In [ ]:
import subprocess

# CONFIG (not a hardcoded path): the dataset's think tags must match the model
# the escape hatch selects ([THINK] for Ministral vs <think> for Qwen).
assert subprocess.run(["python", "-m", "tuned.data.smoke", "--config", CONFIG]).returncode == 0, "dataset build failed"

In [ ]:
# Pre-download the base model in ITS OWN cell. Kaggle batch flushes a cell's
# output only when the cell completes (proven by v6/v7: cancelled runs logged
# nothing from the in-flight training cell) - isolating the download makes this
# phase and its duration visible, and the training child starts from a warm
# HF_HOME cache instead of downloading blind inside the supervisor.
import time
from pathlib import Path as _P

from huggingface_hub import snapshot_download
from tuned.train.config import load_config

_cfg = load_config(CONFIG)
_t0 = time.time()
_snap = snapshot_download(_cfg.model.repo, revision=_cfg.model.revision)
_gb = sum(f.stat().st_size for f in _P(_snap).rglob("*") if f.is_file()) / 1e9
print(f"model snapshot ready: {_gb:.1f} GB in {time.time() - _t0:.0f}s at {_snap}")

In [ ]:
## Green means
- **SAVETEST**: no `# of LoRAs ... does not match` error (unsloth#5677); `last-checkpoint/`
  visible in the private HF checkpoint repo. If it fails twice after the regex scoping,
  switch `CONFIG` above to `configs/law_v1_qwen.yaml` (see runbook in the plan doc).
- **SMOKE**: 60 steps complete, loss trending down, **no NaN** (fp16 canary),
  `peak_vram_gb` < 14. Expected duration 4-6 h - record `approx_tokens_per_sec`
  and total session hours for the main-run plan.
- **RESUME**: run in a *fresh* session; training continues from step 25/50, not step 0.
- Note: SAVETEST touches only ~64 examples - an OOM later in the full SMOKE run is still possible; watch peak_vram_gb.
- **Never cancel the training cell.** Kaggle batch discards a cancelled cell's buffered
  output (v6/v7 lesson) - the watchdog kills and flushes on its own. Mid-run visibility:
  `progress/train.log` in the HF checkpoint repo, refreshed every 5 min.

## Green means
- **SAVETEST**: no `# of LoRAs ... does not match` error (unsloth#5677); `last-checkpoint/`
  visible in the private HF checkpoint repo. If it fails twice after the regex scoping,
  switch `CONFIG` above to `configs/law_v1_qwen.yaml` (see runbook in the plan doc).
- **SMOKE**: 60 steps complete, loss trending down, **no NaN** (fp16 canary),
  `peak_vram_gb` < 14. Expected duration 4-6 h - record `approx_tokens_per_sec`
  and total session hours for the main-run plan.
- **RESUME**: run in a *fresh* session; training continues from step 25/50, not step 0.
- Note: SAVETEST touches only ~64 examples - an OOM later in the full SMOKE run is still possible; watch peak_vram_gb.